## ETL Jobs in Machine Learning Applications

### Data Extraction

In [ ]:
# importing pandas library
import pandas as pd

In [ ]:
# Load the dataset
data = pd.read_csv('train.csv')

In [ ]:
# Display the first few rows of the dataset
data.head()

### Data Transformation

#### Handling Missing Values

In [ ]:
# Fill missing values in 'Age' with the median age
data['Age'].fillna(data['Age'].median(), inplace=True)

# Fill missing values in 'Embarked' with the most common embarkation point
data['Embarked'].fillna(data['Embarked'].mode()[0], inplace=True)

# Drop the 'Cabin' column due to a high number of missing values
data.drop(columns=['Cabin'], inplace=True)

#### Feature Engineering

In [ ]:
# Extract titles from names
data['Title'] = data['Name'].apply(lambda x: x.split(',')[1].split('.')[0].strip())

# Create a family size feature
data['FamilySize'] = data['SibSp'] + data['Parch'] + 1

#### Normalize or scale numerical features and encode categorical features.

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Select numerical and categorical columns
numerical_cols = ['Age', 'Fare', 'FamilySize']
categorical_cols = ['Sex', 'Embarked', 'Title']

# Include 'Survived' in the original DataFrame for transformation
data = data[['Survived'] + numerical_cols + categorical_cols]

# Define the transformers
numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Create a column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

# Apply the transformations
data_transformed = preprocessor.fit_transform(data.drop(columns=['Survived']))

# Get the column names after transformation
transformed_columns = numerical_cols + list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols))

# Convert the transformed data back to a DataFrame for convenience
data_transformed_df = pd.DataFrame(data_transformed.toarray(), columns=transformed_columns)

# Add the 'Survived' column back to the transformed DataFrame
data_transformed_df['Survived'] = data['Survived'].values

# Display the first few rows of the transformed DataFrame
print(data_transformed_df.head())

### Data Loading

#### Storing the clean and transformed data in a relational database.

In [ ]:
from sqlalchemy import create_engine

# Create an SQLite database engine
engine = create_engine('sqlite:///titanic.db')

# Save the transformed DataFrame to a table named 'titanic_transformed'
data_transformed_df.to_sql('titanic_transformed', engine, index=False, if_exists='replace')

In [ ]:
# Function to load data from the database
def load_data_from_db(engine):
    query = "SELECT * FROM titanic_transformed"
    data_from_db = pd.read_sql(query, engine)
    return data_from_db

# Load the data
data_loaded = load_data_from_db(engine)
print(data_loaded.head())

#### Integration with ML Pipeline

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Define the features and target variable
X = data_loaded.drop(columns=['Survived'])
y = data_loaded['Survived']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a logistic regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")